# 07. Yurak kasalliklari xavfini bashorat qilish (Heart Disease Risk Prediction)
### Machine Learning: Pandas, Scikit-Learn (KNN va SVM)

Ushbu loyihada UCI Heart Disease ma'lumotlar to'plami asosida bemorlarning tibbiy ko'rsatkichlari (yosh, qon bosimi, xolesterin, yurak urishi va b.) tahlil qilinib, yurak xastaligi xavfini **K-Nearest Neighbors (KNN)** hamda **Support Vector Machine (SVM)** algoritmlari yordamida bashorat qilish modeli qurilgan.

## 1. Kerakli kutubxonalarni import qilish va ma'lumotlarni yuklash

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

print("Barcha kutubxonalar muvaffaqiyatli yuklandi!")

In [2]:
# Datasetni yuklash
df = pd.read_csv("dataset.csv")
df.columns = df.columns.str.strip().str.replace("﻿", "")
print("Dataset o'lchami:", df.shape)
df.head()

## 2. Dastlabki ma'lumotlar tahlili (EDA - Exploratory Data Analysis)

In [3]:
# Ma'lumotlar tuzilishi va yetishmayotgan qiymatlarni tekshirish
df.info()
print("
Yetishmayotgan qiymatlar soni:", df.isnull().sum().sum())
print("Takroriy qatorlar soni:", df.duplicated().sum())
df.drop_duplicates(inplace=True)
print("Tozalangan dataset o'lchami:", df.shape)

In [4]:
# Asosiy statistik ko'rsatkichlar
df.describe()

In [5]:
# Maqsadli o'zgaruvchi (target) taqsimoti
plt.figure(figsize=(6, 4))
sns.countplot(x="target", data=df, palette=["#10b981", "#ef4444"])
plt.title("Yurak xastaligi xavfi taqsimoti (0: Sog'lom, 1: Xavf bor)")
plt.xlabel("Holat (Target)")
plt.ylabel("Bemorlar soni")
plt.show()

In [6]:
# Xususiyatlar o'rtasidagi korrelyatsiya matritsasi
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Xususiyatlar Korrelyatsiyasi (Heatmap)")
plt.show()

## 3. Ma'lumotlarni tayyorlash: Train / Test ajratish va Masshtablash

In [7]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"O'qitish to'plami (Train): {X_train.shape[0]} ta")
print(f"Sinov to'plami (Test):      {X_test.shape[0]} ta")

## 4. Model 1: K-Nearest Neighbors (KNN)

In [8]:
# KNN modelini o'qitish (k=7, distance weights)
knn = KNeighborsClassifier(n_neighbors=7, weights="distance")
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)
y_prob_knn = knn.predict_proba(X_test_scaled)[:, 1]

print("=== KNN Sinov Natijalari ===")
print(f"Aniqlik (Accuracy): {accuracy_score(y_test, y_pred_knn)*100:.2f}%")
print(f"F1-Score:           {f1_score(y_test, y_pred_knn)*100:.2f}%")
print(f"Precision:          {precision_score(y_test, y_pred_knn)*100:.2f}%")
print(f"Recall:             {recall_score(y_test, y_pred_knn)*100:.2f}%")
print(f"ROC-AUC:            {roc_auc_score(y_test, y_prob_knn)*100:.2f}%")
print("
Klassifikatsiya hisoboti:
", classification_report(y_test, y_pred_knn))

## 5. Model 2: Support Vector Machine (SVM)

In [9]:
# SVM modelini o'qitish (RBF Kernel, C=1.0)
svm = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
y_prob_svm = svm.predict_proba(X_test_scaled)[:, 1]

print("=== SVM (RBF) Sinov Natijalari ===")
print(f"Aniqlik (Accuracy): {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
print(f"F1-Score:           {f1_score(y_test, y_pred_svm)*100:.2f}%")
print(f"Precision:          {precision_score(y_test, y_pred_svm)*100:.2f}%")
print(f"Recall:             {recall_score(y_test, y_pred_svm)*100:.2f}%")
print(f"ROC-AUC:            {roc_auc_score(y_test, y_prob_svm)*100:.2f}%")
print("
Klassifikatsiya hisoboti:
", classification_report(y_test, y_pred_svm))

## 6. Modellar Natijalarini Taqqoslash va Vizualizatsiya

In [10]:
# Confusion Matrixlarni solishtirish
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_svm = confusion_matrix(y_test, y_pred_svm)

sns.heatmap(cm_knn, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title(f"KNN Confusion Matrix (Acc: {accuracy_score(y_test, y_pred_knn)*100:.1f}%)")
axes[0].set_xlabel("Bashorat")
axes[0].set_ylabel("Haqiqiy holat")

sns.heatmap(cm_svm, annot=True, fmt="d", cmap="Greens", ax=axes[1])
axes[1].set_title(f"SVM Confusion Matrix (Acc: {accuracy_score(y_test, y_pred_svm)*100:.1f}%)")
axes[1].set_xlabel("Bashorat")
axes[1].set_ylabel("Haqiqiy holat")
plt.tight_layout()
plt.show()

In [11]:
# ROC egri chiziqlarini taqqoslash
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)

plt.figure(figsize=(7, 5))
plt.plot(fpr_knn, tpr_knn, label=f"KNN (AUC = {roc_auc_score(y_test, y_prob_knn):.3f})")
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {roc_auc_score(y_test, y_prob_svm):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Tasodifiy (AUC = 0.5)")
plt.title("ROC Egri Chizig'i (ROC Curve)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()

## 7. Xulosa va Tavsiyalar
1. **SVM (RBF)** yurak xastaligini aniqlashda eng yuqori aniqlik (**83.61%**) va Recall (**93.94%**) ko'rsatkichini namoyon etdi. Tibbiy tashxislarda bemor kasalligini o'tkazib yubormaslik (Recall) o'ta muhim bo'lganligi sababli SVM modeli ushbu vazifa uchun optimal tanlovdir.
2. **KNN (k=7)** modeli ham barqaror **81.97%** aniqlik va **90.91%** Recall natijasini ko'rsatdi.
3. Barcha parametrlar va masshtablangan qiymatlar Vercel veb-interfeysi uchun  fayliga eksport qilindi.